# K-Way Merge (Merge k Sorted Lists)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Heaps, Linked List · **Difficulty/Frequency:** Very Common (7/10)

> **Related:** [`2. Iterators`](../2.%20Iterators/2.%20Iterators.ipynb) is the *streaming* version of this exact problem — same heap, but exposed as `hasNext()`/`next()` instead of returning a whole list. Study them together.

## Concepts

**What this problem is really testing:**
- Whether you **use the sortedness** you were given, instead of throwing it away and re-sorting
- **Min-heap** as the answer to a repeated "which of these k is smallest?" question
- Pointer surgery on **linked lists**, and the dummy-head trick that makes it painless

**First-principles primer — what is each piece?**

- **Linked list** — nodes each holding a value and a pointer to the next node. You cannot jump to index 7; you can only walk. But you *can* splice a node in or out in O(1) if you already hold a pointer to it — which is exactly what merging needs.
- **Min-heap (priority queue)** — an array-backed binary tree with one rule: every parent ≤ its children. That rule alone guarantees the smallest element sits at index 0, readable in O(1), while push and pop cost O(log size). Python: `heapq.heappush` / `heapq.heappop` on a plain list.
- **Dummy (sentinel) head** — a throwaway node you build the answer behind. Without it, every append needs an `if result is None` special case; with it, `tail.next = node` always works, and the real answer is `dummy.next`.
- **N vs. k** — N is the *total* node count, k is the *number of lists*. Keeping them straight is the whole complexity discussion: N can be 10,000 while k is 3.

**The key insight:**

Each list is already sorted, so the smallest unmerged node overall must be at the **head** of one of the k lists. You never need to look deeper. The only question, repeated N times, is *"which of the k heads is smallest?"*

- Answer it by **scanning** all k heads → O(k) per node → **O(N·k)** total.
- Answer it with a **heap** → O(log k) per node → **O(N log k)** total.

The heap wins because it *remembers* the ordering between rounds instead of rediscovering it from scratch.

**Why not just collect everything and sort?** That is O(N log N) and O(N) extra space. Since `k ≤ N` always, `O(N log k)` is never worse and is dramatically better when k is small — merging 3 lists of 10,000 nodes is `N log 3`, not `N log 10000`.

**The Python gotcha you must mention.** Pushing `(node.val, node)` crashes the moment two nodes tie on value: Python falls through to comparing the second slot, and `ListNode < ListNode` raises `TypeError`. Push `(node.val, i, node)` — the unique list index `i` settles every tie before the node is ever examined.

**Simple worked example.** `lists = [1→4→5, 1→3→4, 2→6]`.

Seed the heap with one node per list: `[(1,0), (1,1), (2,2)]`.

| pop | emit | push back | heap after |
|---|---|---|---|
| `(1,0)` | 1 | `(4,0)` | `(1,1) (2,2) (4,0)` |
| `(1,1)` | 1 | `(3,1)` | `(2,2) (3,1) (4,0)` |
| `(2,2)` | 2 | `(6,2)` | `(3,1) (4,0) (6,2)` |
| `(3,1)` | 3 | `(4,1)` | `(4,0) (4,1) (6,2)` |
| … | … | … | … |

Result: `1→1→2→3→4→4→5→6`. The heap never held more than 3 entries.

## Problem Statement

Given `k` linked lists, each sorted ascending, merge them into one sorted linked list and return its head.

**Example**

```
Input:  [[1,4,5], [1,3,4], [2,6]]
Output: 1->1->2->3->4->4->5->6

Input:  []      -> None
Input:  [[]]    -> None
```

**Constraints**

- `0 <= k <= 10^4`, `0 <= len(lists[i]) <= 500`
- `-10^4 <= value <= 10^4`, total nodes `N <= 10^4`
- every `lists[i]` is already sorted ascending

In [ ]:
from typing import List, Optional


class ListNode:
    def __init__(self, val: int = 0, next: "Optional[ListNode]" = None) -> None:
        self.val = val
        self.next = next

    def __repr__(self) -> str:                  # so failed asserts print something readable
        return f"ListNode({self.val})"


def build(values: List[int]) -> Optional[ListNode]:
    """Python list -> linked list."""
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head


def to_list(head: Optional[ListNode]) -> List[int]:
    """Linked list -> Python list. Guards against cycles from bad pointer surgery."""
    out, seen, node = [], set(), head
    while node is not None:
        if id(node) in seen:
            raise AssertionError("cycle detected in the result list")
        seen.add(id(node))
        out.append(node.val)
        node = node.next
    return out

### Approach 1 — Naive (collect everything, sort, rebuild)

**Idea:** walk every list, dump all N values into an array, sort it, and build a fresh linked list.

It is correct and takes two minutes to write — a perfectly good thing to say out loud as your baseline. But it **discards the sortedness** you were handed, which is the one piece of structure the problem gave you, and it allocates a second copy of every node.

**Time complexity:** **O(N log N)** — dominated by sorting data that was already 90% ordered.

**Space complexity:** O(N) for the array, plus O(N) for the rebuilt nodes.

In [ ]:
def merge_k_naive(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    values = []
    for node in lists:
        while node is not None:
            values.append(node.val)             # flatten everything...
            node = node.next
    values.sort()                               # ...then re-sort what was already sorted
    return build(values)

### Approach 2 — Scan all k heads on every step

**Idea:** keep the heads in an array and, for each output node, scan all k of them to find the smallest.

This is the honest expression of the insight ("the answer is always at some head") without the data structure that makes it cheap. It is a good intermediate step to voice in an interview: it shows you spotted the structure, and it sets up the heap as the fix for a *named* bottleneck.

**Time complexity:** **O(N × k)** — with k = 10⁴ and N = 10⁴ that is 10⁸ comparisons.

**Space complexity:** O(k) for the heads array; the nodes themselves are **relinked, not copied**.

In [ ]:
def merge_k_scan(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    heads = [n for n in lists if n is not None]
    dummy = ListNode()
    tail = dummy
    while heads:
        best = 0
        for i in range(1, len(heads)):          # O(k) linear scan, repeated N times
            if heads[i].val < heads[best].val:
                best = i
        node = heads[best]
        tail.next = node                        # splice the existing node in - no allocation
        tail = node
        if node.next is not None:
            heads[best] = node.next
        else:
            heads.pop(best)                     # that list is finished
    tail.next = None                            # cut the tail loose from its old successor
    return dummy.next

### Approach 3 — Optimal (min-heap of the k heads)

**Idea:** replace the linear scan with a heap. Seed it with one node per non-empty list, then repeat: pop the smallest, splice it onto the result, and push that node's successor.

Three details that matter:

- **The tie-breaker `i`.** `(val, i, node)` — without `i`, equal values make Python compare `ListNode` objects and raise `TypeError`. This is the single most common way this solution fails in a real interview.
- **The heap size is bounded by k, not N.** You only ever push a successor *after* popping its predecessor, so there is at most one entry per list at any moment. That is where O(k) space and O(log k) per operation come from.
- **`tail.next = None` at the end.** The last node spliced in still points at whatever followed it in its *original* list. Leaving that pointer alive can append a stale suffix — or, in the two-pointer variants below, create a cycle. The `to_list` helper above deliberately detects cycles so this bug cannot pass silently.

**Time complexity:** **O(N log k)** — each of the N nodes is pushed once and popped once, at O(log k) each.

**Space complexity:** **O(k)** for the heap. The output reuses the input nodes, so no per-node allocation.

In [ ]:
import heapq


def merge_k_lists(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    heap = []
    for i, node in enumerate(lists):
        if node is not None:                    # skip empty lists at seed time
            heapq.heappush(heap, (node.val, i, node))   # `i` breaks ties -> never compares nodes

    dummy = ListNode()                          # sentinel: no "is the result empty yet?" branch
    tail = dummy
    while heap:
        _, i, node = heapq.heappop(heap)        # O(log k): the global minimum
        tail.next = node
        tail = node
        if node.next is not None:
            heapq.heappush(heap, (node.next.val, i, node.next))   # refill from the SAME list
    tail.next = None                            # cut the stale pointer from the final node
    return dummy.next

### Approach 4 — Divide and conquer (same O(N log k), no heap)

**Idea:** merging *two* sorted lists is easy and needs only a couple of pointers. So merge the k lists in pairs, then merge the results in pairs, and so on — a tournament bracket.

Why it is also O(N log k): there are `log k` rounds, and **every round touches every node at most once**, so each round costs O(N). Total: O(N log k).

Why you might prefer it: no heap, no tuples, no tie-breaker gotcha, and **O(1) extra space** beyond the recursion/loop bookkeeping — the answer to the "can you do it without extra space?" follow-up. Why you might not: the pairing loop is fiddlier to get right under pressure than a heap.

**Time complexity:** **O(N log k)**.

**Space complexity:** **O(1)** extra (iterative pairing); nodes are relinked in place.

In [ ]:
def merge_two(a: Optional[ListNode], b: Optional[ListNode]) -> Optional[ListNode]:
    """Classic two-list merge: relinks existing nodes, allocates nothing but the sentinel."""
    dummy = ListNode()
    tail = dummy
    while a is not None and b is not None:
        if a.val <= b.val:                      # `<=` keeps the merge stable
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a if a is not None else b       # one list is empty; attach the whole remainder
    return dummy.next


def merge_k_divide(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    if not lists:
        return None
    level = list(lists)
    while len(level) > 1:                       # log k rounds, each O(N)
        nxt = []
        for i in range(0, len(level), 2):
            a = level[i]
            b = level[i + 1] if i + 1 < len(level) else None   # odd one out rides along
            nxt.append(merge_two(a, b))
        level = nxt
    return level[0]

## Verification

Run the three examples from the statement, then confirm all four approaches produce the identical merged sequence on randomised input — including the tie-heavy cases that expose the `TypeError` bug and the stale-pointer bug.

In [ ]:
import random

APPROACHES = [merge_k_naive, merge_k_scan, merge_k_lists, merge_k_divide]


def run(fn, lists_of_values):
    """Each approach gets its OWN nodes, since three of the four relink in place."""
    return to_list(fn([build(v) for v in lists_of_values]))


# --- The three examples from the problem statement ---
for fn in APPROACHES:
    assert run(fn, [[1, 4, 5], [1, 3, 4], [2, 6]]) == [1, 1, 2, 3, 4, 4, 5, 6], fn.__name__
    assert run(fn, []) == [], fn.__name__                    # k == 0
    assert run(fn, [[]]) == [], fn.__name__                  # one empty list

# --- Edge cases ---
for fn in APPROACHES:
    assert run(fn, [[], [], []]) == [], fn.__name__          # every list empty
    assert run(fn, [[1]]) == [1], fn.__name__                # single one-node list
    assert run(fn, [[], [2], []]) == [2], fn.__name__        # empties around a real list
    assert run(fn, [[1, 2, 3], [], [4, 5]]) == [1, 2, 3, 4, 5], fn.__name__
    # Very unequal lengths - the short lists drain long before the long one
    assert run(fn, [[1] * 5, list(range(2, 60))]) == [1] * 5 + list(range(2, 60)), fn.__name__
    # Negative values, at the stated constraint bound
    assert run(fn, [[-10000, 0], [-5000, 10000]]) == [-10000, -5000, 0, 10000], fn.__name__

# --- Ties everywhere: this is what raises TypeError without the (val, i, node) tie-breaker ---
for fn in APPROACHES:
    assert run(fn, [[7, 7, 7], [7, 7], [7]]) == [7] * 6, fn.__name__
    assert run(fn, [[0], [0], [0], [0]]) == [0, 0, 0, 0], fn.__name__

# --- All four agree on randomised input; to_list() also proves no cycles were created ---
random.seed(29)
for _ in range(300):
    k = random.randint(0, 8)
    lists_of_values = [
        sorted(random.choices(range(-20, 20), k=random.randint(0, 10))) for _ in range(k)
    ]
    expected = sorted(v for lst in lists_of_values for v in lst)
    for fn in APPROACHES:
        assert run(fn, lists_of_values) == expected, (fn.__name__, lists_of_values)

# --- The optimal version relinks existing nodes rather than allocating new ones ---
originals = [build([1, 4]), build([2, 3])]
first_node_of_list_one = originals[0]
merged = merge_k_lists(originals)
assert merged is first_node_of_list_one, "the result must reuse the input nodes, not copy them"
assert to_list(merged) == [1, 2, 3, 4]

# --- A large tie-heavy case: the heap must never exceed k entries ---
big = [[5] * 200 for _ in range(50)]
assert run(merge_k_lists, big) == [5] * 10000

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **k sorted *arrays* instead of linked lists.** The heap is unchanged; only the cursor changes. Push `(value, list_index, element_index)` and, after popping, push `(arr[li][ei+1], li, ei+1)` if that index exists. Same O(N log k), same O(k) space. Python's `heapq.merge(*arrays)` is exactly this, as a lazy generator.
- **O(1) extra space.** Approach 4 answers this directly: iterative pairwise merging relinks existing nodes and needs only a handful of pointers. Note the heap version is *not* O(1) — the heap is O(k) — so if the interviewer pins you on space, divide-and-conquer is the answer to reach for, not the heap.
- **Streamed lists that do not fit in memory.** The heap already only ever holds k values, so the *merge* is fine; the problem is holding k open streams at once. Merge in batches — combine the first B streams into one intermediate, spill it, repeat, then merge the intermediates. This is exactly the merge phase of an **external sort**, and B is chosen to fit the available buffer.
- **Parallelising.** Divide-and-conquer parallelises naturally: the pairwise merges within one round are fully independent, so each round can be handed to a worker pool. The catch is that the final rounds have very few, very large merges, so speed-up tails off — the last merge alone is O(N) and inherently serial. The heap version does *not* parallelise, since every pop depends on the previous push.
- **Which to pick in a real interview?** Lead with the heap (it is the expected answer and generalises to streaming), then offer divide-and-conquer as the O(1)-space alternative. Naming both, and knowing that they share the same O(N log k), is what separates a pass from a strong pass.

## Empirical complexity check

Hold the **total node count N fixed** and double the **number of lists k**. That isolates the per-element selection cost, which is the only thing that differs between the approaches.

| Growth when k doubles | What it means |
|---|---|
| ~2x | linear in k — the scan re-examines every head for every node |
| ~1x | logarithmic in k — the heap and the divide-and-conquer bracket barely notice |

The naive sort is included as a control: it ignores k entirely, so it should stay flat.

**Reading the numbers honestly:** each timed run also *builds* the k linked lists, which is a fixed O(N) cost independent of k. That constant floor is why the "flat" approaches land near 1.0x rather than exactly 1.0x, and it damps the scan's ratio below a clean 2.0x at small k. The scan's growth still shows through clearly once k is large enough for `N x k` to dominate that floor.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

TOTAL = 20000            # N is FIXED; only k grows


def make_lists(k):
    per = max(1, TOTAL // k)
    # Interleaved arithmetic progressions: each list sorted, heavy overlap between them.
    return ([[i * k + j for i in range(per)] for j in range(k)],)


def run_scan(values):
    merge_k_scan([build(v) for v in values])


def run_heap(values):
    merge_k_lists([build(v) for v in values])


def run_divide(values):
    merge_k_divide([build(v) for v in values])


def run_naive(values):
    merge_k_naive([build(v) for v in values])


benchmark(
    {"Approach 2 - scan all k heads O(N*k)": run_scan,
     "Approach 1 - collect + sort O(N log N)": run_naive,
     "Approach 3 - min-heap O(N log k)": run_heap,
     "Approach 4 - divide & conquer O(N log k)": run_divide},
    make_lists,
    sizes=[16, 32, 64, 128, 256],
    repeats=1,
)

## Patterns learned

- **Never re-sort data that arrives sorted.** Collecting and sorting is O(N log N); merging is O(N log k). Spotting the structure you were handed is usually where the better complexity comes from.
- **A repeated min/max query over a changing set is a heap.** The heap's value is that it *retains* ordering work between queries. Same pattern as Dijkstra, task scheduling, top-K, and the streaming `Iterators` problem next door.
- **Always put a tiebreaker in your heap tuples.** `(key, unique_int, payload)`. It makes ordering deterministic and stops Python from ever comparing payloads that do not support `<`. This one line is the difference between working code and a `TypeError`.
- **Use a dummy head when building a linked list.** It removes every "is this the first node?" branch, and the answer is just `dummy.next`.
- **Relink, do not rebuild.** Splicing existing nodes keeps space at O(1) and preserves object identity — which matters whenever nodes carry more than a value.
- **Cut the tail pointer.** The last node you splice still points into its old list. Setting `tail.next = None` is one line, and forgetting it produces a stale suffix or an infinite cycle — the kind of bug worth writing a cycle check into your test helper for.
- **Two names, two complexities.** Keeping N (total items) and k (number of sources) distinct is what lets you say "O(N log k), which beats O(N log N) whenever k ≪ N" — and that sentence is most of the interview.